In [1]:
import argparse
from pathlib import Path
import glob
from zarr.codecs import BloscCodec

import xarray as xr
import numpy as np

%cd /g/data/w42/dr6273/work/wind_drought/
import src.regions as regions
import src.read as read
import src.utils as utils

/g/data/w42/dr6273/work/wind_drought


In [14]:
def preprocess(ds):
    """ Return smaller region """
    ds = ds.sel(
        longitude=slice(REGION[0], REGION[1]),
        latitude=slice(REGION[2], REGION[3]),
    )
    return ds

ERA5_PATH = "/g/data/rt52/era5/single-levels/reanalysis/" # hourly instantaneous
ERA5_WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/ERA5/"
REGION = regions.get_Aus_boundary()

In [3]:
from dask_setup import setup_dask_client
client, cluster, dask_tmp = setup_dask_client(mode="interactive", workload_type="cpu")   # heavy compute

INFO     [client] Starting Dask client setup (workload_type=cpu | environment=jupyter)
INFO     [resources] Resources detected via PBS (total_cores=104 | total_mem_gib=496.0)


INFO     [client] Temp/spill dir: /jobfs/185036331.gadi-pbs/dask-848996
INFO     [client] Workers: 104 | threads/worker: 1 | processes: True
INFO     [client] Mem: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
INFO     [client] Compression: spill=auto | comm=False
INFO     [client] Dask client ready


[setup_dask_client] Configuration summary
temp/spill dir: /jobfs/185036331.gadi-pbs/dask-848996
Workers: 104 | threads/worker: 1 | processes: True
Memory: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
Compression: spill=auto | comm=False


2026-11-24 13:55:58,549 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c2ea232b0228dbaffddd40ad63b738b2 initialized by task ('rechunk-merge-rechunk-transfer-3c508c12dbc5711abecda74d0901b4d5', 0, 0, 2, 97, 0, 3) executed on worker tcp://127.0.0.1:43195
2026-11-24 13:55:58,796 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 0f5d6a635322828343689c4238ec3373 initialized by task ('rechunk-merge-rechunk-transfer-3c508c12dbc5711abecda74d0901b4d5', 0, 1, 1, 90, 2, 1) executed on worker tcp://127.0.0.1:37847
2026-11-24 13:55:58,805 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3fdd7b1514373c0702faadffd359e941 initialized by task ('rechunk-merge-rechunk-transfer-3c508c12dbc5711abecda74d0901b4d5', 0, 0, 0, 97, 0, 0) executed on worker tcp://127.0.0.1:45413
2026-11-24 13:55:59,333 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 13be5889e3a26e5e2aeac90d0002842e initialized by task ('rechunk-merge-rechunk-transfer-3c508c12dbc5711abecda74d0901b4d5', 

In [38]:
chunks = {"time": -1, "latitude": 25, "longitude": 3}
start=2023
end=2025

In [39]:
variable = "100u"

In [40]:
files = read.era_get_files(ERA5_PATH, variable, start_year=start, end_year=end)

In [41]:
files[:3]

[PosixPath('/g/data/rt52/era5/single-levels/reanalysis/100u/2023/100u_era5_oper_sfc_20230101-20230131.nc'),
 PosixPath('/g/data/rt52/era5/single-levels/reanalysis/100u/2023/100u_era5_oper_sfc_20230201-20230228.nc'),
 PosixPath('/g/data/rt52/era5/single-levels/reanalysis/100u/2023/100u_era5_oper_sfc_20230301-20230331.nc')]

In [42]:
ds = read.open_files(files, preprocess=preprocess)

KeyboardInterrupt: 

In [ ]:
ds

In [31]:
variable = "t2m"

In [32]:
clim_slice=slice("1979", "2025")

In [33]:
ds_chunked = ds[variable].chunk(chunks)

# Daily mean
ds_chunked = ds_chunked.coarsen(time=24).mean() # Hourly to daily mean
ds_chunked["time"] = ds_chunked["time"].dt.floor("D")

# Anomalies
ds_anoms = utils.doy_anom(ds_chunked, clim=clim_slice)

ds_anoms = ds_anoms.chunk({"time": -1, "latitude": -1, "longitude": 8})

In [34]:
ds_anoms

<xarray.DataArray 't2m' (time: 27029, latitude: 141, longitude: 181)> Size: 6GB
dask.array<rechunk-merge, shape=(27029, 141, 181), dtype=float64, chunksize=(27029, 141, 8), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 216kB 1952-01-01 1952-01-02 ... 2025-12-31
    dayofyear  (time) int64 216kB dask.array<chunksize=(27029,), meta=np.ndarray>
  * latitude   (latitude) float32 564B -10.0 -10.25 -10.5 ... -44.5 -44.75 -45.0
  * longitude  (longitude) float32 724B 110.0 110.2 110.5 ... 154.5 154.8 155.0
Attributes:
    units:      K
    long_name:  2 metre temperature

In [35]:
ds = ds_chunked.chunk({"time": -1, "latitude": -1, "longitude": 8})
ds = ds.to_dataset(name=variable)
ds[variable+"_anom"] = ds_anoms

In [36]:
ds

<xarray.Dataset> Size: 11GB
Dimensions:    (longitude: 181, latitude: 141, time: 27029)
Coordinates:
  * longitude  (longitude) float32 724B 110.0 110.2 110.5 ... 154.5 154.8 155.0
  * latitude   (latitude) float32 564B -10.0 -10.25 -10.5 ... -44.5 -44.75 -45.0
  * time       (time) datetime64[ns] 216kB 1952-01-01 1952-01-02 ... 2025-12-31
    dayofyear  (time) int64 216kB dask.array<chunksize=(27029,), meta=np.ndarray>
Data variables:
    t2m        (time, latitude, longitude) float64 6GB dask.array<chunksize=(27029, 141, 8), meta=np.ndarray>
    t2m_anom   (time, latitude, longitude) float64 6GB dask.array<chunksize=(27029, 141, 8), meta=np.ndarray>

In [37]:
encoding = read.specify_encoding(ds)
    
read.write_zarr(
    ds,
    ERA5_WRITE_PATH,
    variable+"_daily_dayofyear_anoms_"+str(1952) + '-' + str(2025)+".zarr",
    encoding
)

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 13.02 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2026-11-24 13:59:01,459 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (m

KeyboardInterrupt: 

2026-11-24 13:59:03,473 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
2026-11-24 13:59:03,477 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
2026-11-24 13:59:03,479 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await